In [16]:
from langchain_classic.chains import LLMChain
from langchain_community.llms import LlamaCpp
from langchain_core.prompts import PromptTemplate

you need to download gguf file


In [2]:
# Make sure the model path is correct for your system!
llm = LlamaCpp(
    model_path=r"C:\Users\trt\models\Phi-3-mini-4k-instruct-fp16.gguf",
    n_gpu_layers=-1,
    max_tokens=500,
    n_ctx=2048,
    seed=42,
    verbose=False
)

Unfortunately, we get no output! As we have seen in previous chapters, Phi-3 requires
a specific prompt template.

In [3]:
llm.invoke("Hi! My name is Maarten. What is 1 + 1?")

''

# A Single Link in the Chain: Prompt Template

The template for Phi-3 is comprised of four main components:
* <s> to indicate when the prompt starts
* <|user|> to indicate the start of the user’s prompt
* <|assistant|> to indicate the start of the model’s output
* <|end|> to indicate the end of either the prompt or the model’s output

In [8]:
# Create a prompt template with the "input_prompt" variable
template = """<s><|user|>
{input_prompt}<|end|>
<|assistant|>"""
prompt = PromptTemplate(
 template=template,
 input_variables=["input_prompt"]
)

In [9]:
basic_chain = prompt | llm

In [ ]:
# Use the chain
basic_chain.invoke(
    {
    "input_prompt": "Hi! My name is Maarten. What is 1 + 1?",
    }
)

c:\Users\trt\Desktop\research_pyt\venv\Lib\site-packages\llama_cpp\llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


' Hello Maarten! The answer to 1 + 1 is 2.'

# A Chain with Multiple Prompts

In [ ]:
# Create a chain for the title of our story
template = """<s><|user|>
Create a title for a story about {summary}. Only return the title.<|end|>
<|assistant|>"""

title_prompt = PromptTemplate(template=template, input_variables=["summary"])

title = LLMChain(llm=llm, prompt=title_prompt, output_key="title")

C:\Users\trt\AppData\Local\Temp\ipykernel_26304\3378294541.py:7: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  title = LLMChain(llm=llm, prompt=title_prompt, output_key="title")


In [18]:
title.invoke({"summary": "a girl that lost her mother"})

c:\Users\trt\Desktop\research_pyt\venv\Lib\site-packages\llama_cpp\llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'summary': 'a girl that lost her mother',
 'title': ' "Whispers of a Farewell: The Journey Through Loss"'}

In [19]:
# Create a chain for the character description using the summary and title
template = """<s><|user|>
Describe the main character of a story about {summary} with the title {title}. 
Use only two sentences.<|end|>
<|assistant|>"""

character_prompt = PromptTemplate(
 template=template, input_variables=["summary", "title"]
)

character = LLMChain(llm=llm, prompt=character_prompt, output_key="character")

In [20]:
# Create a chain for the story using the summary, title, and character description
template = """<s><|user|>
Create a story about {summary} with the title {title}. The main character is: 
{character}. Only return the story and it cannot be longer than one paragraph. 
<|end|>
<|assistant|>"""

story_prompt = PromptTemplate(
 template=template, input_variables=["summary", "title", "character"]
)

story = LLMChain(llm=llm, prompt=story_prompt, output_key="story")

In [21]:
# Combine all three components to create the full chain
llm_chain = title | character | story

In [22]:
llm_chain.invoke("a girl that lost her mother")

c:\Users\trt\Desktop\research_pyt\venv\Lib\site-packages\llama_cpp\llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
c:\Users\trt\Desktop\research_pyt\venv\Lib\site-packages\llama_cpp\llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
c:\Users\trt\Desktop\research_pyt\venv\Lib\site-packages\llama_cpp\llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'summary': 'a girl that lost her mother',
 'title': ' "Lily\'s Lament: A Journey Through Grief"',
 'character': ' Lily, a resilient and compassionate teenager with sparkling blue eyes, is forced to navigate the treacherous waters of grief after losing her mother in a tragic accident. As she embarks on a transformative journey towards healing, her unwavering determination and vulnerability capture the hearts of those around her.',
 'story': " Lily's Lament: A Journey Through Grief\n\nLily, a resilient and compassionate teenager with sparkling blue eyes, found herself lost in the labyrinth of grief after losing her beloved mother to an unforeseeable accident. Her world shattered into fragmented memories; yet, she refused to let despair consume her spirit. With every sunrise, Lily embarked on a transformative journey that led her through the heart-wrenching valleys of sorrow and toward the gentle peaks of healing. Along this path, her unwavering determination to honor her mother's legacy

# Memory: Helping LLMs to Remember Conversations
When we are using LLMs out of the box, they will not remember what was being said in a conversation. You can share your name in one prompt but it will have forgotten it by the next prompt.